In [19]:
#!/usr/bin/env python
"""
Download CMIP6 daily river discharge (rivo, table Eday) for CNRM-ESM2-1,
ensemble r9i1p1f2, for historical, ssp370 and ssp585.

Files are stored in:
    ../data/esgf_downloads/<experiment>/CNRM-ESM2-1/rivo_Eday/
"""

from pathlib import Path
import requests, sys
from bs4 import BeautifulSoup

MODEL      = "CNRM-CERFACS/CNRM-ESM2-1"
ENSEMBLE   = "r9i1p1f2"
TABLE      = "Eday"
VARIABLE   = "rivo"
EXPERIMENTS = ["historical", "ssp370", "ssp585"]

BASE_HOST  = "https://climatedata.umr-cnrm.fr/cnrm/CMIP6/CMIP"

def find_release_folder(url: str) -> str:
    """Return the first dYYYYMMDD/ sub-folder inside 'files/'."""
    html = requests.get(url + "files/").text
    soup = BeautifulSoup(html, "html.parser")
    sub  = next((a["href"] for a in soup.find_all("a") if a["href"].startswith("d")), None)
    if not sub:
        raise RuntimeError("No release folder found under 'files/'")
    return url + "files/" + sub

def list_nc_files(url: str) -> list[str]:
    """Return all .nc filenames in the given HTTP directory."""
    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")
    return [a["href"] for a in soup.find_all("a") if a["href"].endswith(".nc")]

def stream_download(url: str, dest: Path, chunk: int = 1 << 20):
    """Download a file via streaming unless it already exists."""
    if dest.exists():
        print(f"✓ {dest.name} already present")
        return
    print(f"↓ Downloading {dest.name}")
    with requests.get(url, stream=True, timeout=90) as r:
        r.raise_for_status()
        with dest.open("wb") as f:
            for blk in r.iter_content(chunk):
                f.write(blk)

for exp in EXPERIMENTS:
    print(f"\n=== {exp.upper()} ===")
    base_path = f"{BASE_HOST}/{MODEL}/{exp}/{ENSEMBLE}/{TABLE}/{VARIABLE}/gn/"
    try:
        release_url = find_release_folder(base_path)
    except Exception as e:
        print(f"⚠ Could not locate release folder for {exp}: {e}")
        continue

    try:
        nc_files = list_nc_files(release_url)
    except Exception as e:
        print(f"⚠ Failed to list .nc files for {exp}: {e}")
        continue
    if not nc_files:
        print("⚠ No .nc files found – skipping.")
        continue

    out_dir = Path(f"../data/esgf_downloads/{exp}/CNRM-ESM2-1/")
    out_dir.mkdir(parents=True, exist_ok=True)

    for fname in nc_files:
        try:
            stream_download(release_url + fname, out_dir / fname)
        except Exception as e:
            print(f"⚠ Error downloading {fname}: {e}")
            sys.exit(1)

    print(f"✅ Finished {exp}. Files saved to {out_dir}")


=== HISTORICAL ===
↓ Downloading rivo_Eday_CNRM-ESM2-1_historical_r9i1p1f2_gn_18500101-18991231.nc


KeyboardInterrupt: 